# MORTIMER: Multi-asset Optimisation and Risk Targeting with Integrated Monthly E6 Rebalancing

## 1. Research question

Volatility clustering—the persistence in absolute and squared returns—has been documented across financial markets and motivates forecasts of conditional risk (Cont, 2001; Engle, 1982; Bollerslev, 1986). Realized-volatility research uses this persistence to forecast future risk (Andersen et al., 2003), while volatility-managed portfolios apply it to portfolio exposure (Moreira and Muir, 2017). MORTIMER tests whether a lagged volatility forecast can scale a diversified equal-risk-contribution portfolio to reduce net volatility and drawdown relative to an unscaled portfolio, with acceptable return drag and turnover.

The success criteria require at least 10% lower volatility and drawdown, improved conditional tail loss, annual return drag below 150 basis points, annual turnover below two, and risk improvement under doubled costs. Validation rank information coefficient must be positive, and annual folds must support the same conclusion. These criteria make the hypothesis falsifiable.

In [ ]:
import os
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "research/mortimer-provenance.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "research/mortimer-provenance.json").exists():
    raise FileNotFoundError("Open the notebook inside the GQH repository")
os.chdir(ROOT)


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter
from scipy.stats import spearmanr
from IPython.display import Image, display
OUT = Path('results/mortimer-submission')
OUT.mkdir(parents=True, exist_ok=True)
NB_OUT = OUT / 'notebook-evidence'
NB_OUT.mkdir(parents=True, exist_ok=True)
PURPLE = '#6D3B91'
BLUE = '#366A9B'
plt.rcParams.update({'font.size':12,'axes.labelsize':12,'axes.titlesize':12,'xtick.labelsize':11,'ytick.labelsize':11,'legend.fontsize':11})
print('MORTIMER results replay; the held-out out-of-sample (OOS) test is read from its recorded evaluation.')


## 2. Data and universe

The universe comprises six exchange-traded funds (ETFs): QQQ, IWM, HYG, TLT, GLD, and DBC, representing growth and small-cap equities, high-yield credit, Treasury duration, gold, and broad commodities. These are surviving funds rather than point-in-time constituents, so discontinued-fund selection is not measured.

Yahoo adjusted closes include distributions and splits; raw closes and volume support liquidity estimates. Cash accrues from initial-release observations in the Archival Federal Reserve Economic Data (ALFRED) three-month Treasury bill yield series, available on the first exchange session after observation and release. Accrual uses actual days over 365.

The New York Stock Exchange (NYSE) calendar spans 2 January 2008 to 2 October 2026. Before replay, the notebook checks dates, duplicates, positive prices, and rate availability. Missing observations stop the analysis; the history is not filled with future data.

In [ ]:
import hashlib

from dataclasses import asdict, dataclass

from pathlib import Path

import exchange_calendars as xc

import numpy as np

import pandas as pd

from scipy.optimize import minimize

from sklearn.covariance import LedoitWolf

CORE = ("SPY", "TLT", "GLD")

MACRO = ("QQQ", "IWM", "HYG", "TLT", "GLD", "DBC")

@dataclass(frozen=True)
class Variant:
    trial_id: str
    family: str
    target: float = 0.10
    base: str = "equal"
    universe: tuple = CORE
    lookback: int = 63
    threshold: float = 0.05
    cap: float = 1.0
    control: str = "event"
    covariance_days: int = 126
    har_train_min: int = 504

    def parameters(self):
        return asdict(self)

def forecast(history, base, variant, high=None, low=None, state=None):
    portfolio = history @ base
    fast = portfolio.pow(2).ewm(span=20, adjust=False, min_periods=63).mean().iloc[-1]
    slow = portfolio.pow(2).ewm(span=63, adjust=False, min_periods=63).mean().iloc[-1]
    return float(np.sqrt(252 * max(fast, slow)))

def allocation(history, kind, cap):
    n = history.shape[1]
    if kind == "equal":
        return np.full(n, 1 / n)
    covariance = LedoitWolf().fit(history.to_numpy()).covariance_
    scale = max(np.trace(covariance), 1e-12)
    covariance = covariance / scale

    def objective(w):
        if kind == "minvar":
            return float(w @ covariance @ w)
        contribution = w * (covariance @ w)
        return float(np.sum((contribution - contribution.mean()) ** 2))

    fit = minimize(
        objective,
        np.full(n, 1 / n),
        method="SLSQP",
        bounds=[(0, cap)] * n,
        constraints={"type": "eq", "fun": lambda w: w.sum() - 1},
        options={"ftol": 1e-12, "maxiter": 1000},
    )
    if not fit.success or abs(fit.x.sum() - 1) > 1e-07 or (fit.x > cap + 1e-07).any():
        raise RuntimeError(f"Unsupported {kind} allocation: {fit.message}")
    return fit.x

def control(raw, current, variant, streak):
    if variant.control == "monthly":
        return (raw, 0)
    if variant.control == "event":
        return (raw if abs(raw - current) >= variant.threshold else current, 0)
    if raw - current <= -variant.threshold:
        return (raw, 0)
    streak = streak + 1 if raw - current >= 0.1 - 1e-12 else 0
    if streak >= 5:
        return (min(raw, current + 0.1), 0)
    return (current, streak)

def simulate(
    prices,
    rf,
    closes,
    variant,
    high=None,
    low=None,
    cost_bps=5.0,
    borrow_bps=50.0,
    overlay=True,
    static_exposure=None,
    start_position=504,
    guard_returns=None,
    forecast_engine=None,
    allocation_engine=None,
    control_engine=None,
):
    """Close orders execute one session after formation, after that day's return.

    Weights are post-cost NAV fractions. Costs are solved self-consistently
    against drifted dollar holdings, so the cash account and fees reconcile.
    """
    forecast_engine = forecast_engine or forecast
    allocation_engine = allocation_engine or allocation
    control_engine = control_engine or control
    prices = prices.loc[:, list(variant.universe)]
    if (
        prices.isna().any().any()
        or not prices.index.is_monotonic_increasing
        or prices.index.has_duplicates
    ):
        raise ValueError(
            "Invalid price history: missing, duplicate or unordered observations"
        )
    returns = prices.pct_change(fill_method=None)
    returns.iloc[0] = 0.0
    rf = rf.reindex(prices.index)
    if rf.isna().any() or (prices <= 0).any().any():
        raise ValueError("Missing available risk-free rate or invalid prices")
    n = prices.shape[1]
    holdings = np.zeros(n)
    cash = 1.0
    base = np.full(n, 1 / n)
    pending = None
    exposure = 0.0
    streak = 0
    har_state = {}
    rows, orders = [], []
    weight_rows, base_rows, trade_rows, forecast_weight_rows = [], [], [], []
    previous_month = None
    for t, date in enumerate(prices.index):
        old_nav = holdings.sum() + cash
        risky = float(holdings @ returns.iloc[t].to_numpy())
        cash_pnl = cash * rf.iloc[t] if cash >= 0 else 0.0
        financing = (
            -cash
            * (
                rf.iloc[t]
                + borrow_bps / 10000 / 365 * (date - prices.index[t - 1]).days
            )
            if cash < 0 and t
            else 0.0
        )
        holdings *= 1 + returns.iloc[t].to_numpy()
        cash += cash_pnl - financing
        before_fee = holdings.sum() + cash
        trade = np.zeros(n)
        fee = turnover = 0.0
        if pending is not None:
            desired, signal_date, signal_base, signal_exposure = pending
            # Solve fee = c * sum(abs(post_fee_nav*target - drifted_holdings)).
            after_fee = before_fee
            for _ in range(50):
                updated = (
                    before_fee
                    - cost_bps / 10000 * np.abs(after_fee * desired - holdings).sum()
                )
                if abs(updated - after_fee) < 1e-14:
                    break
                after_fee = updated
            after_fee = updated
            trade = after_fee * desired - holdings
            fee = before_fee - after_fee
            old_weights = np.r_[holdings / before_fee, cash / before_fee]
            holdings = after_fee * desired
            cash = after_fee - holdings.sum()
            new_weights = np.r_[desired, 1 - desired.sum()]
            turnover = 0.5 * np.abs(new_weights - old_weights).sum()
            exposure = signal_exposure
            base = signal_base
            effective = closes.iloc[t + 1] if t + 1 < len(prices) else pd.NaT
            assert closes.loc[signal_date] < closes.loc[date]
            orders.append(
                {
                    "signal_timestamp": closes.loc[signal_date],
                    "feature_timestamp": closes.loc[signal_date],
                    "execution_timestamp": closes.loc[date],
                    "effective_from_timestamp": closes.loc[date],
                    "first_return_end_timestamp": effective,
                    "exposure": exposure,
                }
            )
            pending = None
        nav = holdings.sum() + cash
        if nav <= 0:
            raise RuntimeError("Insolvent simulation")
        rows.append(
            {
                "net_return": nav / old_nav - 1,
                "gross_return": (risky + cash_pnl - financing) / old_nav,
                "risky_contribution": risky / old_nav,
                "cash_contribution": cash_pnl / old_nav,
                "financing_cost": financing / old_nav,
                "transaction_cost": fee / old_nav,
                "traded_notional": np.abs(trade).sum() / before_fee,
                "turnover": turnover,
                "equity": nav,
                "exposure": holdings.sum() / nav,
                "forecast": np.nan,
                "raw_exposure": np.nan,
            }
        )
        weight_rows.append(holdings / nav)
        base_rows.append(base.copy())
        forecast_weight_rows.append(base.copy())
        trade_rows.append(trade / before_fee)
        if t < start_position or t == len(prices) - 1:
            continue
        month = date.to_period("M")
        # Monthly signal on last actual NYSE session of month, not first session.
        month_end = prices.index[t + 1].to_period("M") != month
        first = previous_month is None
        base_change = first or month_end
        next_base = (
            allocation_engine(
                returns.iloc[max(1, t - variant.covariance_days + 1) : t + 1],
                variant.base,
                0.35 if n == 6 else 0.70,
            )
            if base_change
            else base
        )
        history = returns.iloc[1 : t + 1]
        # HAR is only registered for the constant equal-weight base.
        vol = forecast_engine(
            history,
            next_base,
            variant,
            None if high is None else high.loc[:date, prices.columns],
            None if low is None else low.loc[:date, prices.columns],
            har_state,
        )
        rows[-1]["forecast"] = vol
        forecast_weight_rows[-1] = next_base.copy()
        if not np.isfinite(vol) or vol <= 0:
            continue
        raw = min(variant.cap, variant.target / vol) if overlay else 1.0
        if static_exposure is not None:
            raw = static_exposure
        if variant.cap > 1 and overlay and static_exposure is None:
            trailing = guard_returns.loc[:date].tail(252)
            if len(trailing) < 252 or (1 + trailing).prod() <= 1:
                raw = min(raw, 1.0)
        rows[-1]["raw_exposure"] = raw
        if variant.control == "monthly" and not (base_change or first):
            continue
        actual = holdings.sum() / nav
        desired_exposure, streak = (
            control_engine(raw, actual, variant, streak)
            if overlay and static_exposure is None
            else (raw, 0)
        )
        if first:
            desired_exposure, streak = raw, 0
        # Hard leverage ceiling overrides slow restoration/ordinary hysteresis.
        if variant.cap > 1 and raw <= 1 and actual > 1:
            desired_exposure = min(desired_exposure, 1.0)
        if first or base_change or abs(desired_exposure - actual) > 1e-10:
            desired_exposure = min(variant.cap, max(0.0, desired_exposure))
            pending = (next_base * desired_exposure, date, next_base, desired_exposure)
        previous_month = month
    result = pd.DataFrame(rows, index=prices.index)
    result.attrs["orders"] = pd.DataFrame(orders)
    result.attrs["weights"] = pd.DataFrame(
        weight_rows, index=prices.index, columns=prices.columns
    )
    result.attrs["base_weights"] = pd.DataFrame(
        base_rows, index=prices.index, columns=prices.columns
    )
    result.attrs["forecast_weights"] = pd.DataFrame(
        forecast_weight_rows, index=prices.index, columns=prices.columns
    )
    result.attrs["trades"] = pd.DataFrame(
        trade_rows, index=prices.index, columns=prices.columns
    )
    return result

def metrics(result, rf):
    r = result.net_return
    if len(r) < 2:
        raise ValueError("Insufficient metric observations")
    excess = r - rf.reindex(r.index)
    wealth = pd.concat(
        [pd.Series([1.0]), (1 + r).cumprod().reset_index(drop=True)], ignore_index=True
    )
    dd = wealth / wealth.cummax() - 1
    cagr = float(wealth.iloc[-1] ** (252 / len(r)) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(252))
    downside = float(np.sqrt(np.mean(np.minimum(excess, 0) ** 2)) * np.sqrt(252))
    var = float(r.quantile(0.05))
    tail = r[r <= var]
    durations, duration = [], 0
    for value in dd:
        duration = duration + 1 if value < -1e-12 else 0
        durations.append(duration)
    return {
        "cagr": cagr,
        "volatility": vol,
        "sharpe": excess.mean() * 252 / vol if vol else np.nan,
        "sortino": excess.mean() * 252 / downside if downside else np.nan,
        "max_drawdown": float(dd.min()),
        "calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "var_95": var,
        "cvar_95": float(tail.mean()),
        "worst_1d": float(r.min()),
        "worst_5d": float((1 + r).rolling(5).apply(np.prod, raw=True).min() - 1),
        "worst_20d": float((1 + r).rolling(20).apply(np.prod, raw=True).min() - 1),
        "turnover": result.turnover.sum() * 252 / len(r),
        "implementation_cost": result.transaction_cost.sum() * 252 / len(r),
        "terminal_wealth": float(wealth.iloc[-1]),
        "recovery_sessions": max(durations),
        "unrecovered_at_end": bool(dd.iloc[-1] < -1e-12),
    }

def load_data(cache):
    cache = Path(cache)
    market = cache / "market-20080102-20241002.parquet"
    rates_path = cache / "rates-20080102-20241002.parquet"
    expected_hashes = [
        "e5f957a2b78b4a08cede2183cd59012f73beaffc38f0821d43bb458df08be222",
        "5eb2ada6d8d007969c111064ae6c5de089e8bd61c0156cb49cc98484cea2b5d9",
    ]
    for path, digest in zip([market, rates_path], expected_hashes):
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise ValueError(f"Data hash differs: {path.name}")
    panel = pd.read_parquet(market)
    rates = pd.read_parquet(rates_path)
    cal = xc.get_calendar("XNYS", start="2007-01-01", end="2026-12-31")
    planned = cal.sessions_in_range("2008-01-02", "2026-10-02").tz_localize(None)
    boundary = max(
        planned[int(len(planned) * 0.8)],
        planned[planned >= pd.Timestamp("2024-10-02")][0],
    )
    sessions = planned[planned < boundary]
    assert panel.index.max() < boundary
    prices = panel.xs("Adj Close", axis=1, level=1)[list(MACRO)].reindex(sessions)
    assert not prices.isna().any().any() and not prices.index.has_duplicates
    releases = rates.sort_values(["realtime_start", "date"]).copy()
    releases["available"] = [
        planned[planned > max(d, r)][0]
        for d, r in zip(releases.date, releases.realtime_start)
    ]
    releases = releases.groupby("available").tail(1)
    merged = pd.merge_asof(
        pd.DataFrame({"date": sessions}),
        releases.sort_values("available"),
        left_on="date",
        right_on="available",
    )
    assert merged.value.notna().all()
    rate = pd.Series(merged.value.to_numpy() / 100, index=sessions)
    elapsed = sessions.to_series().diff().dt.days.fillna(1)
    rf = (1 + rate.shift().fillna(rate.iloc[0])) ** (elapsed / 365) - 1
    closes = pd.Series([cal.session_close(d) for d in sessions], index=sessions)
    return panel, prices, rf, closes, str(boundary.date()), expected_hashes

E6_VARIANT = Variant("ERC", "ensemble", base="erc", universe=MACRO)

## 3. Portfolio construction

Let $\widehat\Sigma_t$ denote the 126-session Ledoit–Wolf covariance estimate. The monthly constrained risk-balanced base solves

$$\min_b\sum_{i=1}^{6}\left[b_i(\widehat\Sigma_t b)_i-\frac{b^\top\widehat\Sigma_t b}{6}\right]^2,
\qquad \mathbf1^\top b=1,\quad 0\leq b_i\leq0.35.$$

**Risk contribution** is $b_i(\widehat\Sigma_t b)_i$ in variance units. Equal-risk contribution (ERC) allocates similar risk contributions; binding caps can prevent exact equality. Ledoit–Wolf **covariance shrinkage** reduces finite-sample estimation noise by combining the sample covariance with a structured target.

## 4. Volatility forecast and exposure

An exponentially weighted moving average (EWMA) assigns geometrically declining weights to past squared returns. For a fixed current base $b_t$, compute historical portfolio returns $x_s=b_t^\top r_s$ using observations through signal close $t$. The forecast is

$$\widehat\sigma_t=\sqrt{252\max\{\operatorname{EWMA}_{20}(x_s^2),\operatorname{EWMA}_{63}(x_s^2)\}},\qquad
k_t^{raw}=\min\left(1,\frac{0.10}{\widehat\sigma_t}\right).$$

Taking the larger of the 20- and 63-span estimates combines a faster response with a slower risk measure. A five-percentage-point change triggers an exposure trade; monthly base rebalances proceed independently. Gross exposure cannot exceed one.

## 5. Execution and transaction costs

Information through close $t$ forms an order filled at close $t+1$, after that session's return. New holdings first earn the following close-to-close return. Positions drift between fills, so the 35% limit applies to base weights rather than continuously realised weights.

For pre-fee risky dollar holdings $h_i$, pre-fee net asset value (NAV) $V$, desired post-fee weights $w_i$, and execution charge $c=0.0005$, solve

$$V^{post}=V-c\sum_i|V^{post}w_i-h_i|.$$

Costs are five basis points of risky dollars bought plus sold, covering spread, commission, and closing slippage. Turnover is half the absolute change in risky and cash weights. A ten-basis-point stress is reported for development data; the held-out test contains one evaluation at five basis points. Cash transfers are free, and the exposure cap removes borrowing costs.

The simulator records signals, fills, positions, gross returns, cash accrual, fees, and net returns separately. Hand-computable cases verify timing and self-financing accounting.

## 6. Research design

The 550-session warm-up precedes the scored sample. Training ends on 31 December 2018; validation runs from 1 January 2019 through 1 October 2024. The held-out out-of-sample (OOS) period runs from 2 October 2024 through 2 October 2026, the shorter of the most recent 20% of sessions and two years. Annual validation folds preserve chronological order and trading state.

The replay reads recorded returns for MORTIMER and matched unscaled ERC, checks the final boundary, and verifies development arithmetic. Strategy and reproduction functions appear in notebook cells for inspection.

In [ ]:
import argparse

import hashlib

import json

import sys

from pathlib import Path

import exchange_calendars as xc

import numpy as np

import pandas as pd

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_files(expected):
    for name, digest in expected.items():
        path = Path(name)
        if not path.is_file():
            raise FileNotFoundError(f"Restore the registered immutable asset: {path}")
        if file_hash(path) != digest:
            raise ValueError(f"Registered evidence hash differs: {path}")

def load_recorded_inputs():
    """Align prices, cash rates, and exchange closes on the observed calendar."""
    provenance = json.loads(Path("research/mortimer-provenance.json").read_text())
    verify_files(provenance["input_sha256"])
    panel, prices, rf, _, boundary, _ = load_data("data/cache/erc")
    final = pd.read_parquet("data/cache/erc-final/market.parquet")
    rates = pd.read_parquet("data/cache/erc-final/rates.parquet")
    cal = xc.get_calendar("XNYS", start="2007-01-01", end="2026-12-31")
    sessions = cal.sessions_in_range("2008-01-02", "2026-10-02").tz_localize(None)
    full = pd.concat(
        [prices, final.xs("Adj Close", axis=1, level=1)[list(MACRO)].loc[boundary:]]
    ).reindex(sessions)
    if full.isna().any().any() or full.index.has_duplicates:
        raise ValueError("Missing or duplicate recorded common price observations")
    rates["available"] = [
        sessions[sessions > max(d, r)][0] if (sessions > max(d, r)).any() else pd.NaT
        for d, r in zip(rates.date, rates.realtime_start)
    ]
    rates = (
        rates.dropna(subset=["available"])
        .sort_values(["available", "date"])
        .groupby("available")
        .tail(1)
    )
    final_sessions = sessions[sessions >= pd.Timestamp(boundary)]
    merged = pd.merge_asof(
        pd.DataFrame({"date": final_sessions}),
        rates.sort_values("available"),
        left_on="date",
        right_on="available",
    )
    if merged.value.isna().any():
        raise ValueError("Unavailable initial-release cash observations")
    yield_rate = pd.Series(merged.value.to_numpy() / 100, index=final_sessions)
    elapsed = sessions.to_series().diff().dt.days.loc[final_sessions]
    final_rf = (1 + yield_rate.shift().fillna(yield_rate.iloc[0])) ** (
        elapsed / 365
    ) - 1
    rf = pd.concat([rf, final_rf])
    closes = pd.Series([cal.session_close(d) for d in sessions], index=sessions)
    actual_boundary = max(
        sessions[int(len(sessions) * 0.8)],
        sessions[sessions >= sessions[-1] - pd.DateOffset(years=2)][0],
    )
    if actual_boundary != pd.Timestamp(boundary):
        raise ValueError("Recorded holdout boundary differs from observed history")
    full_panel = pd.concat([panel.loc[:, list(MACRO)], final.loc[boundary:]])
    return full_panel, full, rf, closes, boundary

def read_daily(path):
    return pd.read_csv(path, index_col=0, parse_dates=True)

def sample_masks(index, boundary):
    return {
        "train": index.year <= 2018,
        "validation": (index.year >= 2019) & (index < pd.Timestamp(boundary)),
        "oos_test": index >= pd.Timestamp(boundary),
    }

def reproduce(output=Path("results/mortimer-submission"), reuse=True):
    """Verify recorded series and rebuild deterministic result tables.

    The default path reads saved returns. Development replay checks arithmetic
    before the out-of-sample (OOS) boundary and never simulates the held-out period.
    """
    output = Path(output)
    provenance = json.loads(Path("research/mortimer-provenance.json").read_text())
    verify_files(provenance["input_sha256"])
    verify_files(provenance["recorded_sha256"])
    verify_files(provenance["scientific_source_sha256"])
    receipt = output / "reproduction-receipt.json"
    if reuse and receipt.exists():
        previous = json.loads(receipt.read_text())
        verify_files(previous["output_sha256"])
        if previous["provenance_sha256"] != file_hash(
            "research/mortimer-provenance.json"
        ):
            raise ValueError("Reproduction provenance changed")
    output.mkdir(parents=True, exist_ok=True)
    _panel, prices, rf, closes, boundary = load_recorded_inputs()
    rows, annual, datasets = [], [], {}
    for strategy in ("E6", "ERC"):
        dev = read_daily(f"results/archive/erc-guarded-v1/{strategy}-daily.csv")
        final = read_daily(f"results/archive/erc-final-v1/{strategy}-daily.csv")
        frame = pd.concat([dev, final])
        if frame.index.has_duplicates or not frame.index.equals(prices.index[550:]):
            raise ValueError("Recorded scoring sample differs from common warm-up")
        datasets[strategy] = frame
        frame.to_csv(output / f"{strategy}-daily.csv")
        for sample, mask in sample_masks(frame.index, boundary).items():
            current = frame.loc[mask]
            rows.append(
                dict(
                    strategy=strategy,
                    sample=sample,
                    cost_bps=5,
                    observations=len(current),
                    first=str(current.index[0].date()),
                    last=str(current.index[-1].date()),
                    **metrics(current, rf),
                )
            )
        for year, current in frame.groupby(frame.index.year):
            annual.append(dict(strategy=strategy, year=year, **metrics(current, rf)))
        for kind in ("weights", "base_weights", "trades", "orders"):
            # Individual-asset final holdings were not recorded in this archive.
            # Keep the development-only evidence explicitly bounded.
            recorded = Path(f"results/archive/erc-guarded-v1/{strategy}-{kind}.csv")
            (output / f"{strategy}-development-{kind}.csv").write_bytes(
                recorded.read_bytes()
            )
    performance = pd.DataFrame(rows)
    recorded_performance = pd.concat(
        [
            pd.read_csv("results/archive/erc-guarded-v1/performance.csv"),
            pd.read_csv("results/archive/erc-final-v1/performance.csv"),
        ]
    )
    recorded_performance["sample"] = recorded_performance["sample"].replace(
        {"reused_validation": "validation", "reused_final": "oos_test"}
    )
    for row in rows:
        registered = recorded_performance[
            (recorded_performance.strategy == row["strategy"])
            & (recorded_performance["sample"] == row["sample"])
        ].iloc[0]
        for metric in ("cagr", "volatility", "sharpe", "max_drawdown", "turnover"):
            if not np.isclose(row[metric], registered[metric], rtol=1e-10, atol=1e-12):
                raise ValueError(
                    f"Recorded metric mismatch: {row['strategy']} {row['sample']} {metric}"
                )
    performance.to_csv(output / "performance.csv", index=False)
    pd.DataFrame(annual).to_csv(output / "annual.csv", index=False)
    rf.rename("cash_return").to_csv(output / "cash-returns.csv")
    prices.to_csv(output / "market-adjusted-prices.csv")
    stress_rows = []
    original = Path("results/20261004T003708-e3127963")
    for strategy, filename in (
        ("E6", "E6-double_cost-3fc474039b.csv"),
        ("ERC", "E6-double_cost_base-811cdde690.csv"),
    ):
        stress = read_daily(original / filename).iloc[550:]
        stress.to_csv(output / f"{strategy}-10bp-development-daily.csv")
        for sample, mask in sample_masks(stress.index, boundary).items():
            if mask.any():
                stress_rows.append(
                    dict(
                        strategy=strategy,
                        sample=sample,
                        cost_bps=10,
                        **metrics(stress.loc[mask], rf),
                    )
                )
    pd.DataFrame(stress_rows).to_csv(output / "doubled-cost.csv", index=False)
    forecast_file = original / "tables/forecast_diagnostics.csv"
    forecast = pd.read_csv(forecast_file)
    forecast[forecast.trial_id == "E6"].to_csv(output / "forecast-ic.csv", index=False)
    if not reuse:
        dev_prices = prices.loc[:"2024-10-01"]
        replay = simulate(dev_prices, rf, closes, E6_VARIANT, start_position=550).iloc[
            550:
        ]
        saved = datasets["E6"].loc[:"2024-10-01"]
        np.testing.assert_allclose(
            replay.to_numpy(), saved.to_numpy(), rtol=1e-10, atol=1e-12, equal_nan=True
        )
    generated = [
        output / name
        for name in (
            "performance.csv",
            "annual.csv",
            "doubled-cost.csv",
            "forecast-ic.csv",
            "cash-returns.csv",
            "market-adjusted-prices.csv",
            "E6-daily.csv",
            "ERC-daily.csv",
            "E6-10bp-development-daily.csv",
            "ERC-10bp-development-daily.csv",
        )
    ]
    receipt.write_text(
        json.dumps(
            {
                "mode": "verified recorded evidence consolidation"
                if reuse
                else "recorded evidence plus development arithmetic replay",
                "provenance_sha256": file_hash("research/mortimer-provenance.json"),
                "scored_start": str(prices.index[550].date()),
                "boundary": boundary,
                "final_status": "held-out OOS test evaluated once after specification freeze",
                "final_10bp_status": "not recorded; no new final simulation",
                "output_sha256": {str(p): file_hash(p) for p in generated},
            },
            indent=2,
        )
        + "\n"
    )
    return performance

In [ ]:
replay_summary = reproduce(reuse=True)
print(replay_summary)


## 7. Data quality

Coverage, return distributions, cross-asset correlations, and lagged dollar volume describe the sample.

### Price history and return distribution

Price indices, daily return distributions, cross-asset correlations, and coverage summarise the observed sample.

In [ ]:
def save_figure(fig, name):
    fig.savefig(NB_OUT / (name+'.pdf'))
    fig.savefig(NB_OUT / (name+'.png'), dpi=150)
    display(Image(filename=str(NB_OUT / (name+'.png'))))
    plt.close(fig)

def centred_legend(fig, axes, ncol=2):
    handles, labels = axes.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5,0.995), ncol=ncol, frameon=False, fontsize=11)

def mark_sample_periods(ax, dates):
    start, end = dates.min(), dates.max()
    periods = [
        ('Training', start, pd.Timestamp('2018-12-31')),
        ('Validation', pd.Timestamp('2019-01-01'), pd.Timestamp('2024-10-01')),
        ('Test', pd.Timestamp('2024-10-02'), end),
    ]
    for boundary in (pd.Timestamp('2019-01-01'), pd.Timestamp('2024-10-02')):
        ax.axvline(boundary, color='#68717A', linestyle='--', linewidth=.8)
    for label, period_start, period_end in periods:
        ax.text(period_start + (period_end-period_start)/2, .96, label,
                transform=ax.get_xaxis_transform(), ha='center', va='top',
                fontsize=9, color='#4D5661',
                bbox={'facecolor':'white','edgecolor':'none','alpha':.8,'pad':1})

# Cached market inputs are inspected without acquisition.
dev_panel = pd.read_parquet('data/cache/erc/market-20080102-20241002.parquet')
final_panel = pd.read_parquet('data/cache/erc-final/market.parquet')
observed = pd.concat([dev_panel,final_panel]).sort_index()
observed = observed.loc[~observed.index.duplicated(keep='last')]
prices = pd.read_csv(OUT/'market-adjusted-prices.csv',index_col=0,parse_dates=True).reindex(columns=MACRO)
asset_returns = prices.pct_change(fill_method=None).dropna()
coverage = pd.DataFrame({'observed_sessions':prices.notna().sum(),'missing_sessions':prices.isna().sum(),'positive_prices':(prices>0).all()})
display(coverage)
display(asset_returns.describe().T)
coverage.to_csv(NB_OUT/'coverage.csv')
fig,axes = plt.subplots(1,2,figsize=(11,4))
for ticker in MACRO: axes[0].plot(prices.index,prices[ticker]/prices[ticker].iloc[0],label=ticker)
axes[0].set(xlabel='Session date',ylabel='Adjusted price index (start = 1)')
centred_legend(fig,axes[0],ncol=6)
axes[0].set_xticks(pd.to_datetime(['2008-01-01','2014-01-01','2020-01-01','2026-01-01']))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
correlation=asset_returns.corr()
im=axes[1].imshow(correlation,vmin=-1,vmax=1,cmap='RdBu_r')
axes[1].set_xticks(range(6),MACRO,rotation=40)
axes[1].set_yticks(range(6),MACRO)
fig.colorbar(im,ax=axes[1],label='Return correlation')
fig.subplots_adjust(left=.08,right=.92,bottom=.20,top=.82,wspace=.35)
save_figure(fig,'observed-data')
fig,ax=plt.subplots(figsize=(8,4))
for ticker in MACRO:ax.hist(asset_returns[ticker],bins=80,density=True,histtype='step',label=ticker)
ax.set(xlabel='Daily adjusted return',ylabel='Density')
ax.xaxis.set_major_formatter(PercentFormatter(1))
centred_legend(fig,ax,ncol=6)
fig.subplots_adjust(left=.11,right=.89,bottom=.16,top=.82)
save_figure(fig,'return-distributions')


## 8. Performance and uncertainty

We calculate annualised return as compound annual growth rate (CAGR) from split-local net wealth using 252 trading sessions per year. Annualised volatility is the sample standard deviation of daily returns multiplied by $\sqrt{252}$, and the Sharpe ratio uses daily excess returns relative to the cash return proxy (Sharpe, 1994). Maximum drawdown is measured from initial wealth of one. Conditional value at risk (CVaR) is the mean return in the lower 5% tail. Annual turnover is half the L1 change in risky and cash weights. Results are net of transaction costs and reported separately for training, validation, and the held-out out-of-sample (OOS) test against matched unscaled ERC. Equity curves are normalised to one at each split's start; the drawdown and turnover series are plotted alongside them.

In [ ]:
performance=pd.read_csv(OUT/'performance.csv')
performance_display=performance.copy()
performance_display['strategy']=performance_display['strategy'].replace({'E6':'MORTIMER'})
display(performance_display.round(5))
cost_display=pd.read_csv(OUT/'doubled-cost.csv')
if 'strategy' in cost_display: cost_display['strategy']=cost_display['strategy'].replace({'E6':'MORTIMER'})
display(cost_display.round(5))
daily={name:pd.read_csv(OUT/(name+'-daily.csv'),index_col=0,parse_dates=True) for name in ['E6','ERC']}
fig,axes=plt.subplots(2,3,figsize=(12,6))
splits=[('Training',None,'2018-12-31'),('Validation','2019-01-01','2024-10-01'),('Out-of-sample test','2024-10-02',None)]
for col,(label,start,end) in enumerate(splits):
    for name,color in [('E6',PURPLE),('ERC',BLUE)]:
        frame=daily[name].loc[start:end]
        wealth=(1+frame.net_return).cumprod()
        drawdown=wealth/np.maximum.accumulate(np.r_[1,wealth])[1:]-1
        axes[0,col].plot(frame.index,wealth,label='MORTIMER' if name=='E6' else 'Unscaled ERC',color=color)
        axes[1,col].plot(frame.index,drawdown,color=color)
    axes[0,col].set_title(label)
    axes[0,col].set_ylabel('Net equity (start = 1)')
    axes[1,col].set(xlabel='Session date',ylabel='Drawdown')
    axes[1,col].yaxis.set_major_formatter(PercentFormatter(1))
    ticks=pd.to_datetime(['2010-01-01','2014-01-01','2018-01-01'] if col==0 else ['2019-01-01','2021-01-01','2023-01-01'] if col==1 else ['2024-10-02','2025-10-01','2026-10-02'])
    for row in range(2):
        axes[row,col].set_xticks(ticks)
        axes[row,col].xaxis.set_major_formatter(mdates.DateFormatter('%Y' if col<2 else '%Y-%m'))
        axes[row,col].grid(alpha=.2)
centred_legend(fig,axes[0,0])
fig.subplots_adjust(left=.08,right=.92,bottom=.13,top=.87,wspace=.42,hspace=.38)
save_figure(fig,'split-net-evidence')


## 9. Forecast evaluation

The **information coefficient (IC)** is the correlation between the volatility forecast and subsequent realised variance; the rank IC uses ranked observations. The calibration plot compares the forecast with subsequent 21-session risk on the drifting unscaled portfolio.

Monthly returns, rolling volatility, asset weights, and realised exposure characterise the portfolio through time. Asset-level holdings are available for development; aggregate exposure is reported for the OOS period.

In [ ]:
e6=daily['E6']
monthly=(1+e6.net_return).resample('ME').prod()-1
heat=pd.DataFrame({'year':monthly.index.year,'month':monthly.index.month,'net_return':monthly.to_numpy()}).pivot(index='year',columns='month',values='net_return')
fig,axes=plt.subplots(2,1,figsize=(9,7))
im=axes[0].imshow(heat,aspect='auto',cmap='RdBu',vmin=-.06,vmax=.06)
axes[0].set_xticks(range(12),['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[0].set_yticks(range(len(heat)),heat.index)
axes[0].set(xlabel='Month',ylabel='Year')
fig.colorbar(im,ax=axes[0],format=PercentFormatter(1),label='Monthly net return')
for name,color in [('E6',PURPLE),('ERC',BLUE)]:
    rolling=daily[name].net_return.rolling(126).std()*np.sqrt(252)
    axes[1].plot(rolling.index,rolling,label='MORTIMER' if name=='E6' else 'Unscaled ERC',color=color)
axes[1].set(xlabel='Session date',ylabel='126-session net volatility')
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
mark_sample_periods(axes[1],daily['E6'].index)
axes[1].legend(loc='upper center',bbox_to_anchor=(.5,1.13),ncol=2,frameon=False)
fig.subplots_adjust(left=.10,right=.90,bottom=.09,top=.96,hspace=.35)
save_figure(fig,'monthly-and-rolling-risk')
fig,axes=plt.subplots(2,1,figsize=(9,5))
axes[0].plot(e6.index,e6.exposure,color=PURPLE,label='MORTIMER')
axes[0].set(ylabel='Realised risky exposure',ylim=(0,1.05))
for name,color in [('E6',PURPLE),('ERC',BLUE)]:
    axes[1].plot(daily[name].index,daily[name].turnover.rolling(252).sum(),color=color,label='MORTIMER' if name=='E6' else 'Unscaled ERC')
axes[1].set(xlabel='Session date',ylabel='Trailing annual turnover')
for ax in axes:
    mark_sample_periods(ax,daily['E6'].index)
centred_legend(fig,axes[0],ncol=1)
axes[1].legend(loc='upper center',bbox_to_anchor=(.5,1.17),ncol=2,frameon=False)
fig.subplots_adjust(left=.14,right=.86,bottom=.13,top=.84,hspace=.38)
save_figure(fig,'exposure-and-turnover')
heat.to_csv(NB_OUT/'monthly-net-returns.csv')
# Following-window risk is a retrospective diagnostic, never a trading feature.
future=(daily['ERC'].net_return.pow(2).rolling(21).mean().shift(-21)*252).pow(.5)
calibration=pd.DataFrame({'forecast':e6.forecast,'following_unscaled_risk':future}).dropna()
calibration.to_csv(NB_OUT/'risk-calibration.csv')
fig,ax=plt.subplots(figsize=(8,4))
ax.scatter(calibration.forecast,calibration.following_unscaled_risk,s=8,alpha=.25,color=PURPLE)
ax.set(xlabel='Signal-close fast/slow volatility forecast',ylabel='Following 21-session\nunscaled realised risk')
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.subplots_adjust(left=.15,right=.85,bottom=.19,top=.94)
save_figure(fig,'risk-calibration')
weights_path=OUT/'E6-development-weights.csv'
if weights_path.exists():
    weights=pd.read_csv(weights_path,index_col=0,parse_dates=True).loc[e6.index[0]:]
    fig,ax=plt.subplots(figsize=(9,4))
    ax.stackplot(weights.index,weights.T,labels=weights.columns)
    ax.set(xlabel='Session date',ylabel='Post-cost risky weight',ylim=(0,1.05))
    centred_legend(fig,ax,ncol=6)
    fig.subplots_adjust(left=.13,right=.87,bottom=.16,top=.80)
    save_figure(fig,'development-asset-weights')
    display(pd.Series({'maximum_realised_instrument_weight':weights.max().max(),'maximum_realised_gross_exposure':weights.sum(axis=1).max(),'mean_cash_weight':(1-weights.sum(axis=1)).mean()}))


## 10. Cost sensitivity

## 11. Factor exposure

## 12. Liquidity and capacity

A ten-basis-point cost case applies to development data. Paired block-bootstrap intervals quantify uncertainty in validation return differences.

Average daily dollar volume (ADDV) is calculated from lagged volume and closing prices. Participation is risky trade notional $Q$ divided by ADDV. The impact scenario is $I=Y\sigma\sqrt{Q/ADDV}$, with $Y=0.5$ and lagged volatility. The capital table reports maximum participation and impact-adjusted returns by initial net asset value (NAV).

In [ ]:
costs=pd.read_csv(OUT/'doubled-cost.csv')
costs['strategy']=costs['strategy'].replace({'E6':'MORTIMER'})
display(costs.round(5))
display(pd.read_csv(OUT/'paired-bootstrap.csv').round(5))


In [ ]:
factors=pd.read_csv(OUT/'factor-attribution.csv')
display(factors.round(5))


In [ ]:
capacity=pd.read_csv(OUT/'capacity.csv')
display(capacity.round(5))


## 13. Conclusion

The validation results show lower volatility and drawdown alongside a 15-basis-point annual return difference. In the held-out out-of-sample (OOS) test, risk reductions are smaller and the Sharpe ratio is lower than for unscaled ERC, indicating that the overlay moderates portfolio risk without improving risk-adjusted return. Further work can extend the cost sensitivity with point-in-time fund histories, transaction-level execution data, and observed market-impact estimates.

## References

See the [research specification](../research/hypothesis.md), [README](../README.md), and [quant note](../report/quant-note.pdf). Volatility persistence and forecasting are studied by [Cont (2001)](https://doi.org/10.1080/713665670), [Engle (1982)](https://doi.org/10.2307/1912773), [Bollerslev (1986)](https://doi.org/10.1016/0304-4076(86)90063-1), and [Andersen et al. (2003)](https://doi.org/10.1111/1468-0262.00418). Portfolio methods include [Ledoit and Wolf (2004)](https://doi.org/10.1016/S0047-259X(03)00096-4), [Maillard, Roncalli, and Teïletche (2010)](https://doi.org/10.3905/jpm.2010.36.4.060), [Moreira and Muir (2017)](https://doi.org/10.1111/jofi.12513), and [DeMiguel, Garlappi, and Uppal (2009)](https://doi.org/10.1093/rfs/hhm075). The Sharpe ratio follows [Sharpe (1994)](https://doi.org/10.3905/jpm.1994.409501); the impact scenario draws on [Tóth et al. (2011)](https://doi.org/10.1103/PhysRevX.1.021006). Data sources are [ALFRED](https://fred.stlouisfed.org/docs/api/fred/series_observations.html), the [Federal Reserve three-month Treasury yield series](https://fred.stlouisfed.org/series/DGS3MO), [Kenneth French's factor library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), and [Yahoo Finance](https://finance.yahoo.com/).